In [2]:
import numpy as np
import torch

vocab_size = 8
embedding_dim = 3

# Numpy
# Embedding Matrizen erzeugen
U_np = np.random.randn(vocab_size, embedding_dim)
V_np = np.random.randn(vocab_size, embedding_dim)

# Beispiel auswählen
# Index target word
t = 0
# Index correct context word
o = 1

# target Vektor erzeugen
u = U_np[t]

# v^T * u
scores = V_np @ u
# Softmax
p = np.exp(scores) / np.sum(scores)

# target Vektor
y = np.zeros(vocab_size)
y[o] = 1.0

# Gradient nach V
grad_V_np = np.outer(y - p, u)
# Gradient nach u
grad_u_np = V_np.T @ (y - p)

# Pytorch
U_torch = torch.tensor(U_np, dtype=torch.float64, requires_grad=True)
V_torch = torch.tensor(V_np, dtype=torch.float64, requires_grad=True)

# target Vektor
u_torch = U_torch[t]
scores_torch = V_torch @ u_torch
log_softmax = torch.log_softmax(scores_torch, dim=0)

target = log_softmax[o]
target.backward()

grad_V_torch = V_torch.grad.detach().numpy()
grad_U_torch = U_torch.grad.detach().numpy()

print("NumPy grad_v_o:")
print(grad_V_np[o])

print("PyTorch grad_v_o:")
print(grad_V_torch[o])

print("NumPy grad_u:")
print(grad_u_np)

print("PyTorch grad_u:")
print(grad_U_torch[t])

NumPy grad_v_o:
[-1.11537779  2.62198412 -0.47447138]
PyTorch grad_v_o:
[-0.90914513  2.13718089 -0.38674191]
NumPy grad_u:
[ 3.54712184  7.27556956 -0.72621625]
PyTorch grad_u:
[ 1.72338067 -0.61585796 -1.47340853]


In [3]:
# Problem: teure Berechnung von Softmax für jedes Wort
# Lösung: Negative Sampling, da man nur noch für k+1 Wörter berechnen muss anstatt über ganz V

In [8]:
# Embedding Matrizen erzeugen
U = np.random.randn(vocab_size, embedding_dim)
V = np.random.randn(vocab_size, embedding_dim)

neg_samples = [2, 5, 7]

u = U[t]
grad_u_softmax = V.T @ (y - p)
grad_V_softmax = np.outer(y - p, u)


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


v_o = V[o]

score_pos = u @ v_o
sig_pos = sigmoid(score_pos)

grad_u_ns = (1 - sig_pos) * v_o
grad_V_ns = np.zeros_like(V)
grad_V_ns[o] = (1 - sig_pos) * u

for n in neg_samples:
    v_n = V[n]
    score_neg = u @ v_n
    sig_neg = sigmoid(score_neg)
    
    grad_u_ns -= sig_neg * v_n
    grad_V_ns[n] = -sig_neg * u

print("softmax:", p[o])
print("negative sampling:", sig_pos)

softmax: -0.19373557434441654
negative sampling: 0.8312653466717246


In [ ]:
def softmax(logits: np.ndarray) -> np.ndarray:
    """Numerically stable softmax (subtract max before exp)."""
    subtracted_max = logits - np.max(logits)
    exp = np.exp(subtracted_max)
    return exp / np.sum(exp)

def temperature_scale(logits: np.ndarray, tau: float) -> np.ndarray:
    """Return logits / tau."""
    return logits / tau

def top_k_filter(probs: np.ndarray, k: int) -> np.ndarray:
    """Zero out all probabilities outside the top-k; renormalise to sum to 1."""
    # Array mit 0en gleiche Form wie probs
    filtered = np.zeros_like(probs)
    # Indizes top_k Werte
    top_k = np.argsort(probs)[-k:]
    # top_k bekommen ursprüngliche Werte zurück
    filtered[top_k] = probs[top_k]
    # renormalisieren
    return filtered / np.sum(filtered)

def top_p_filter(probs: np.ndarray, p: float) -> np.ndarray:
    """
    Nucleus (top-p) filter.
    Sort by descending probability; keep the smallest prefix whose cumulative
    mass >= p; zero the rest; renormalise.
    """
    # sortieren, sodass größte Wahrscheinlichkeit als erstes
    sorted_id = np.argsort(probs)[::-1]
    # probs addieren
    cumulative = np.cumsum(probs[sorted_id])
    # alle Werte behalten bis überschreiten
    keep = cumulative <= p
    # ersten Wert nach überschreiten mitnehmen
    keep[np.argmax(cumulative >= p)] = True
    # alle nicht erlaubten auf 0 setzen
    filtered = np.zeros_like(probs)
    # keep behalten
    filtered[sorted_id[keep]] = probs[sorted_id[keep]]
    # renormalisieren
    return filtered / np.sum(filtered)

def sample_token(probs: np.ndarray) -> int:
    """Sample one token index proportional to probs."""
    return np.random.choice(len(probs), p=probs)

In [6]:
logits = np.array([2.0, 1.0, 0.5, 0.1, -0.5, -1.0, -2.0])

probs = softmax(temperature_scale(logits, tau=1.0))
print("probs      :", probs.round(3))
# Expected: [0.529 0.195 0.118 0.079 0.043 0.026 0.010]

probs_sharp = softmax(temperature_scale(logits, tau=0.5))
print("tau=0.5    :", probs_sharp.round(3))
# More peaked: index 0 dominates more strongly.

probs_flat = softmax(temperature_scale(logits, tau=2.0))
print("tau=2.0    :", probs_flat.round(3))
# More uniform: gap between indices shrinks.

filtered_k = top_k_filter(probs, k=3)
print("top-3      :", filtered_k.round(3))
# Expected: [0.629 0.231 0.140 0.    0.    0.    0.   ]

filtered_p = top_p_filter(probs, p=0.9)
print("top-p=0.9  :", filtered_p.round(3))
# Tokens 0-3 survive (cumulative mass 0.921 >= 0.9); renormalised.

np.random.seed(42)
samples = [sample_token(top_p_filter(
               softmax(temperature_scale(logits, 0.8)), 0.9))
           for _ in range(10)]
print("samples    :", samples)
# Indices drawn from {0,1,2,3}; index 0 most frequent.

probs      : [0.529 0.195 0.118 0.079 0.043 0.026 0.01 ]
tau=0.5    : [0.822 0.111 0.041 0.018 0.006 0.002 0.   ]
tau=2.0    : [0.321 0.195 0.152 0.124 0.092 0.072 0.044]
top-3      : [0.629 0.231 0.14  0.    0.    0.    0.   ]
top-p=0.9  : [0.575 0.211 0.128 0.086 0.    0.    0.   ]
samples    : [0, 3, 1, 0, 0, 0, 0, 2, 0, 1]


In [ ]:
from typing import Callable, List

def beam_search(
    forward_fn: Callable[[List[int]], np.ndarray],
    initial_token: int,
    beam_width: int,
    max_len: int,
    stop_token: int = -1,
) -> List[int]:
    beams = [([initial_token], 0.0)]
    """
    Beam search decoder.

    forward_fn    -- callable: takes a list of token ids (the current
                     hypothesis) and returns a 1-D logit array of size |V|.
    initial_token -- id of the seed / BOS token.
    beam_width    -- number of hypotheses to keep at each step.
    max_len       -- maximum tokens to generate (not counting initial_token).
    stop_token    -- if >= 0, a hypothesis is complete when it generates this
                     token; decoding stops once all beams are complete or
                     max_len is reached.

    Returns the token list of the best complete hypothesis,
    excluding initial_token. Scores are accumulated log-probabilities.
    """

    for i in range(max_len):
        candidates = []

        for tokens, score in beams:
            # wenn fertig nicht mehr erweitern
            if stop_token >= 0 and tokens[-1] == stop_token:
                candidates.append((tokens, score))
                continue

            logits = forward_fn(tokens)
            probs = softmax(logits)
            log_probs = np.log(probs)

            # für jedes mögliche nächste token Sequenz bauen und score addieren
            for token_id, log_prob in enumerate(log_probs):
                candidates.append((tokens + [token_id], score + log_prob))

        # nach score sortieren
        candidates.sort(key=lambda x: x[1], reverse=True)
        # nur besten beam_width behalten
        beams = candidates[:beam_width]

        # wenn alle beams fertig, dann abbrechen
        if stop_token >= 0 and all(tokens[-1] == stop_token for tokens, score in beams):
            break

    best_tokens, best_score = beams[0]
    return best_tokens[1:]

In [23]:
# Vocabulary: 0=START  1="the"  2="cat"  3="sat"  4="dog"  5="ran"  6=STOP
ID2WORD = ["START", "the", "cat", "sat", "dog", "ran", "STOP"]

TRANSITIONS = np.array([
    [-10,  2.0,  0.5, -10,  1.5, -10, -10],  # START → the / cat / dog
    [-10,  -10,  2.5, -10,  2.0, -10, -10],  # the  → cat / dog
    [-10,  -10,  -10,  3.0, -10,  1.0,  0.5],# cat  → sat / ran / STOP
    [-10,  -10,  -10, -10,  -10, -10,  3.0], # sat  → STOP
    [-10,  -10,  -10, -10,  -10,  2.5,  0.5],# dog  → ran / STOP
    [-10,  -10,  -10, -10,  -10, -10,  3.0], # ran  → STOP
    [-10,  -10,  -10, -10,  -10, -10, -10],  # STOP (terminal)
], dtype=float)

def lm_forward(token_ids: List[int]) -> np.ndarray:
    return TRANSITIONS[token_ids[-1]]

result = beam_search(lm_forward, initial_token=0,
                     beam_width=2, max_len=6, stop_token=6)
print([ID2WORD[t] for t in result])
# Expected: ["the", "cat", "sat", "STOP"]

result_greedy = beam_search(lm_forward, initial_token=0,
                            beam_width=1, max_len=6, stop_token=6)
print([ID2WORD[t] for t in result_greedy])
# k=1 is greedy: also ["the", "cat", "sat", "STOP"]

['dog', 'ran', 'STOP']
['the', 'cat', 'sat', 'STOP']
